# 02 — Proditec: HPO log preprocessing

Loads the raw Optuna SQLite log from notebook 01 (study `pdt_public_patchcore_hpo_logs_v1.0.1`,
2,999 COMPLETE trials), filters/cleans it, and writes the surrogate-ready table
(`surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv` — the filename keeps its
original `_xgb` suffix from an earlier prototyping phase; the surrogate trained on it is
CatBoost, see notebook 03) consumed by every downstream Proditec notebook and by the demo app.
Implements Section 3.3 of the paper.


# Optuna's logs preprocessing

In [ ]:
from pathlib import Path
import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
DB_PATH = Path("outputs_patchcore/pdt_public_patchcore_hpo_logs.db")
STORAGE = f"sqlite:///{DB_PATH.as_posix()}"
STUDY_NAME = "pdt_public_patchcore_hpo_logs_v1.0.1"

In [ ]:
study = optuna.load_study(study_name=STUDY_NAME, storage=STORAGE)
print("loaded:", study.study_name, "| trials:", len(study.trials), "| best:", study.best_value)

df = study.trials_dataframe(attrs=("number","value","state","params","user_attrs"))
df = df[df["state"] == "COMPLETE"].copy()
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = df.dropna(subset=["value"])
df.head()


In [ ]:
s = df["value"]
print("n complete:", len(df))
print(s.describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]))


In [ ]:
s = df["value"].to_numpy()

plt.figure(figsize=(7,4))
plt.hist(s, bins=50)
plt.title("F1 distribution (all complete trials)")
plt.xlabel("F1"); plt.ylabel("count")
plt.tight_layout(); plt.show()

plt.figure(figsize=(7,4))
plt.hist(s[s >= max(0.0, np.quantile(s, 0.7))], bins=35)
plt.title("F1 distribution (zoom on upper tail)")
plt.xlabel("F1"); plt.ylabel("count")
plt.tight_layout(); plt.show()


In [ ]:
best_so_far = df.sort_values("number")["value"].cummax()

plt.figure(figsize=(7,4))
plt.plot(df.sort_values("number")["number"], best_so_far)
plt.title("Best F1 so far vs trial number")
plt.xlabel("trial"); plt.ylabel("best F1 so far")
plt.tight_layout(); plt.show()


In [ ]:
# pick a few params you care about (edit this list freely)
params_to_plot = [
    "params_backbone",
    "params_layers_key",
    "params_pre_trained",
    "params_image_size_key",
    "params_center_crop_key",
    "params_reduction",
    "params_batch_size",
    "params_max_patches_per_image",
]

for col in params_to_plot:
    if col not in df.columns:
        continue
    g = df[[col, "value"]].dropna()
    # keep plot readable (avoid very large category counts)
    if g[col].nunique() > 15:
        continue

    cats = list(g[col].astype(str).unique())
    data = [g.loc[g[col].astype(str)==c, "value"].values for c in cats]

    plt.figure(figsize=(8,4))
    plt.boxplot(data, labels=cats, showfliers=False)
    plt.title(f"F1 by {col.replace('params_','')}")
    plt.ylabel("F1")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
best_row = df.loc[df["value"].idxmax()].copy()
print("best trial:", int(best_row["number"]), "| F1:", float(best_row["value"]))

best_params = {k.replace("params_", ""): best_row[k] for k in df.columns if k.startswith("params_")}
best_attrs  = {k.replace("user_attrs_", ""): best_row[k] for k in df.columns if k.startswith("user_attrs_")}

print("\nBest params:")
for k,v in best_params.items():
    print(f"  {k}: {v}")


In [ ]:
df.describe()

In [ ]:
df.head().T

In [ ]:
df.count()

In [ ]:
cols_to_drop = [
    "user_attrs_soft_train_fraction_q01",
    "user_attrs_f1_val_at_budget",
    "number",
    "state",
]

df = df.drop(columns=cols_to_drop, errors="ignore")


In [ ]:
df["params_soft_train_fraction"] = df["params_soft_train_fraction"].round(2)
df["params_coreset_sampling_ratio"] = df["params_coreset_sampling_ratio"].round(3)
df["params_soft_review_budget"] = df["params_soft_review_budget"].round(3)


In [ ]:
df.head().T

In [ ]:
# -----------------------------
# 1) Define target + features
# -----------------------------
TARGET = "value"

FEATURES = [
    "params_backbone",
    "params_batch_size",
    "params_center_crop_key",
    "params_coreset_sampling_ratio",
    "params_image_size_key",
    "params_layers_key",
    "params_max_patches_per_image",
    "params_num_neighbors",
    "params_pre_trained",
    "params_reduction",
    "params_soft_corruption_level",
    "params_soft_review_budget",
    "params_soft_train_fraction",
]

keep_cols = [TARGET] + FEATURES


In [ ]:
# -----------------------------
# 2) Subset + basic cleaning
# -----------------------------
df_export = df.loc[:, keep_cols].copy()

# Drop rows with missing values in any kept column
df_export = df_export.dropna(subset=keep_cols)


In [ ]:
# -----------------------------
# 3) Enforce stable dtypes
# -----------------------------
# Treat these as categorical strings (important for one-hot later)
cat_as_str = [
    "params_backbone",
    "params_center_crop_key",
    "params_image_size_key",           # looks numeric but is a key -> treat as categorical
    "params_layers_key",
    "params_pre_trained",              # bool -> stringify for stable one-hot
    "params_reduction",
    "params_soft_corruption_level",
    "params_batch_size",               # {8,16} but categorical choice
    "params_max_patches_per_image",    # {128,256,512,1024} categorical choice
]
for c in cat_as_str:
    df_export[c] = df_export[c].astype(str)

# Ensure numeric columns are numeric
num_cols = ["params_coreset_sampling_ratio", "params_num_neighbors", "params_soft_review_budget", "params_soft_train_fraction", TARGET]
for c in num_cols:
    df_export[c] = pd.to_numeric(df_export[c], errors="coerce")

df_export = df_export.dropna(subset=num_cols)

In [ ]:
# -----------------------------
# 5) Export to CSV
# -----------------------------
out_path = "surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"
df_export.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Shape:", df_export.shape)
df_export.head()